In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv("../data/cookie_cats.csv")
g30 = df[df.version == "gate_30"]
g40 = df[df.version == "gate_40"]

for metric in ["retention_7", "retention_1"]:
    a, b = g40[metric], g30[metric]
    z, p = proportions_ztest([a.sum(), b.sum()], [len(a), len(b)])
    diff = a.mean() - b.mean()
    se = np.sqrt(a.mean()*(1-a.mean())/len(a) + b.mean()*(1-b.mean())/len(b))
    print(metric, "разница, п.п.:", round(diff*100, 3),
          "| z =", round(z, 3), "| p =", round(p, 4),
          "| 95% ДИ, п.п.:", round((diff-1.96*se)*100, 3), round((diff+1.96*se)*100, 3))

retention_7 разница, п.п.: -0.82 | z = -3.164 | p = 0.0016 | 95% ДИ, п.п.: -1.328 -0.312
retention_1 разница, п.п.: -0.591 | z = -1.784 | p = 0.0744 | 95% ДИ, п.п.: -1.239 0.058


In [2]:
rng = np.random.default_rng(42)
B = 10000
a = g40["retention_7"].to_numpy().astype(int)
b = g30["retention_7"].to_numpy().astype(int)

diffs = np.empty(B)
for i in range(B):
    diffs[i] = rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean()

print("бутстреп ДИ, п.п.:", np.percentile(diffs, [2.5, 97.5]) * 100)

бутстреп ДИ, п.п.: [-1.31916566 -0.32880152]


In [3]:
a = g40["sum_gamerounds"].to_numpy()
b = g30["sum_gamerounds"].to_numpy()
print(pd.Series(a).describe(), pd.Series(b).describe(), sep="\n\n")

rng = np.random.default_rng(42)
diffs = np.array([rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean()
                  for _ in range(5000)])
print("разница средних:", a.mean() - b.mean())
print("бутстреп ДИ:", np.percentile(diffs, [2.5, 97.5]))

count    45489.000000
mean        51.298776
std        103.294416
min          0.000000
25%          5.000000
50%         16.000000
75%         52.000000
max       2640.000000
dtype: float64

count    44700.000000
mean        52.456264
std        256.716423
min          0.000000
25%          5.000000
50%         17.000000
75%         50.000000
max      49854.000000
dtype: float64
разница средних: -1.157488453953249
бутстреп ДИ: [-4.01674468  0.99774324]
